In [123]:
import pandas as pd
import re

df = pd.read_csv('original.csv', encoding='utf-8-sig', low_memory=False)
print(f"원본: {len(df)}행")

원본: 4942행


In [124]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4942 entries, 0 to 4941
Columns: 136 entries, 연번 to 한국대상여부
dtypes: float64(126), int64(1), object(9)
memory usage: 5.1+ MB


### 1. 불필요 규제 형태 제거
- 우회 수출 : 관세청 데이터로 알 수 없음

In [125]:
# 우회수출 전체 제거
df = df[~df['규제형태(진행상황)'].str.contains('우회수출', na=False)]

# 규제형태 정리 (괄호 앞만 추출)
df['regulation_type'] = df['규제형태(진행상황)'].str.extract(r'^([^(]+)')
df['regulation_type'] = df['regulation_type'].str.strip()

# 규제상태 정리
df['regulation_status'] = df['규제형태(진행상황)'].str.extract(r'\(([^)]+)\)')
df['regulation_status'] = df['regulation_status'].str.strip()

print(f"우회수출 제거 후: {len(df)}행")

우회수출 제거 후: 4688행


In [126]:
df['regulation_status'].value_counts()

regulation_status
규제중       3876
조사중        778
초국경보조금       2
Name: count, dtype: int64

In [127]:
df['regulation_type'].value_counts()

regulation_type
반덤핑      3810
상계관세      733
세이프가드     113
Name: count, dtype: int64

### 2. 규제시행국 EU 회원국 통합

In [128]:
eu_member_codes = [
    'AT','BE','BG','CZ','DE','DK','ES','FI','FR','GR',
    'HR','HU','IE','IT','LT','LU','LV','MT','NL','PL',
    'PT','RO','SE','SI','SK'
]
df['규제시행국'] = df['규제시행국'].replace(eu_member_codes, 'EU')

In [129]:
# EU인 경우
eu_count = (df['규제시행국'] == 'EU').sum()

# EU가 아닌 경우
non_eu_count = (df['규제시행국'] != 'EU').sum()

print("EU:", eu_count)
print("Non-EU:", non_eu_count)


EU: 2940
Non-EU: 1748


In [130]:
non_eu = df.loc[df['규제시행국'] != 'EU', '규제시행국'].unique()
print(non_eu)

['AE' 'AM' 'AR' 'AU' 'AZ' 'BD' 'BR' 'BY' 'CA' 'CH' 'CI' 'CL' 'CN' 'CO'
 'CR' 'CU' 'DO' 'DZ' 'EC' 'EG' 'ET' 'GB' 'GH' 'GT' 'HK' 'ID' 'IL' 'IN'
 'IQ' 'IR' 'JO' 'JP' 'KE' 'KG' 'KH' 'KW' 'KZ' 'LA' 'LK' 'MA' 'MG' 'MM'
 'MN' 'MX' 'MY' 'MZ' 'NG' 'NZ' 'OM' 'PA' 'PE' 'PH' 'PK' 'PY' 'QA' 'RS'
 'RU' 'SA' 'SD' 'SG' 'TH' 'TN' 'TR' 'TW' 'TZ' 'UA' 'US' 'UZ' 'VE' 'VN'
 'ZA' 'ZW']


In [131]:
# EU 규제 중복 제거
df = df.drop_duplicates(subset=[
    '규제시행국',           # 추가 ← 이게 핵심
    '품목명',
    '규제대상국 ',
    '최종 판정결과(부과기간)'
])
print(f"중복 제거 후: {len(df)}행")

중복 제거 후: 1801행


### 3. HS 코드 칼럼 정제

In [132]:
# HS 코드 칼럼 melt
hs_cols = [c for c in df.columns if 'HS_코드' in c]
id_cols = ['연번', '규제시행국', '품목명', 'regulation_type',
           'regulation_status', '규제대상국 ',
           '최종 판정결과(부과기간)', '최종 판정결과(관세율)',
           '한국대상여부']

df_melt = df[id_cols + hs_cols].melt(
    id_vars=id_cols,
    value_vars=hs_cols,
    value_name='hs_code'
).dropna(subset=['hs_code'])

df_melt = df_melt.drop(columns=['variable'])
print(f"HS코드 melt 후: {len(df_melt)}행")

HS코드 melt 후: 10457행


In [133]:
# HS 코드 칼럼 4단위로 통일
df_melt['hs_code'] = df_melt['hs_code'].astype(str).str.extract(r'(\d+)')[0]
df_melt['hs_4digit'] = df_melt['hs_code'].str[:4]

In [134]:
# HS 72류만 필터 (철강)
df_melt = df_melt[df_melt['hs_4digit'].str.startswith('72')]
print(f"HS 72류 필터 후: {len(df_melt)}행")

HS 72류 필터 후: 3822행


### 4. 규제 대상국 분리

In [135]:
df_melt['규제대상국 '] = df_melt['규제대상국 '].str.split(',')
df_melt = df_melt.explode('규제대상국 ')
df_melt['규제대상국 '] = df_melt['규제대상국 '].str.strip()
df_melt = df_melt.rename(columns={'규제대상국 ': 'target_country'})
print(f"규제대상국 explode 후: {len(df_melt)}행")

규제대상국 explode 후: 8949행


In [136]:
# 불필요한 행 삭제
remove_targets = ['전세계', '', '기타국', '기타국(한국 포함)', 'UAE베트남', '일본 중국']
df_melt = df_melt[~df_melt['target_country'].isin(remove_targets)]
df_melt = df_melt[df_melt['target_country'].notna()]
print(f"불필요 행 제거 후: {len(df_melt)}행")

불필요 행 제거 후: 8483행


In [137]:
countries = df_melt['target_country'].value_counts().index.tolist()
print(countries)

['중국', '한국', '인도', '대만', '베트남', '튀르키예', '인도네시아', '일본', '러시아', '브라질', '우크라이나', '영국', 'EU', '이탈리아', '남아공', '태국', '벨라루스', '멕시코', '말레이시아', '독일', '이집트', '캐나다', '호주', '네덜란드', '이란', '벨기에', '스페인', '카자흐스탄', '덴마크', '프랑스', 'UAE', '몰도바', '오스트리아', '포르투갈', '튀르기예', '싱가폴', '알제리아', '불가리', '홍콩', '오만', '루마니아', '유럽', '베네수엘라', '알제리', '슬로바키아', '아르헨티나', '스웨덴', '미국', '싱가포르', '그리스', '라트비아', '폴란드', '세르비야', '코스타리카', '그루지야', '트리니다드 토바고', '터키', '핀란드']


In [138]:
# 이름 통일
df_melt['target_country'] = df_melt['target_country'].replace({
    '튀르기예': '튀르키예',
    '터키':    '튀르키예',
    '싱가폴':  '싱가포르',
    '알제리아': '알제리',
    '유럽':    'EU',
    '세르비야': '세르비아',
})

In [139]:
countries = df_melt['target_country'].value_counts().index.tolist()
print(countries)

['중국', '한국', '인도', '대만', '튀르키예', '베트남', '인도네시아', '일본', '러시아', '브라질', '우크라이나', 'EU', '영국', '이탈리아', '남아공', '태국', '벨라루스', '멕시코', '말레이시아', '독일', '캐나다', '이집트', '네덜란드', '호주', '이란', '벨기에', '스페인', '알제리', '카자흐스탄', '싱가포르', '덴마크', '프랑스', 'UAE', '몰도바', '오스트리아', '포르투갈', '오만', '불가리', '홍콩', '루마니아', '베네수엘라', '아르헨티나', '슬로바키아', '스웨덴', '미국', '그리스', '세르비아', '라트비아', '폴란드', '코스타리카', '그루지야', '트리니다드 토바고', '핀란드']


### 5. 한국 대상 여부 int 값 변경

In [140]:
df_melt['is_korea_target'] = (df_melt['한국대상여부'] == 'Y').astype(int)
print(f"\nis_korea_target 분포:")
print(df_melt['is_korea_target'].value_counts())


is_korea_target 분포:
is_korea_target
0    4629
1    3854
Name: count, dtype: int64


### 6. 날짜 처리

In [150]:
def extract_dates(text):
    if pd.isna(text):
        return None, None, 'NaN'

    text = str(text)

    # 연-월-일 패턴과 연-월 패턴 둘 다 찾기
    dates = re.findall(r'\d{4}-\d{2}(?:-\d{2})?', text)

    if '부과기간' in text:
        start = dates[0] if len(dates) > 0 else None
        end   = dates[1] if len(dates) > 1 else None
        return start, end, 'confirmed'

    elif '조사개시' in text:
        start = dates[0] if len(dates) > 0 else None
        return start, None, 'investigating'

    else:
        start = dates[0] if len(dates) > 0 else None
        end   = dates[1] if len(dates) > 1 else None
        return start, end, 'unknown'

df_melt[['start_date','end_date','date_type']] = df_melt['최종 판정결과(부과기간)'].apply(
    lambda x: pd.Series(extract_dates(x))
)

# 연-월만 있는 경우 일(day)을 1로 채워서 파싱
# 수정: format='mixed' 추가
df_melt['start_date'] = pd.to_datetime(df_melt['start_date'], errors='coerce', format='mixed')
df_melt['end_date']   = pd.to_datetime(df_melt['end_date'],   errors='coerce', format='mixed')
df_melt['start_ym']   = df_melt['start_date'].dt.to_period('M')

### 7. 컬럼 정리

date_type과 regulation_status가 불일치 하는 경우가 5건 있는데(연번 645) 무관하긴 할테지만 일단 남겨두고 두 컬럼 모두 살리겠음

In [152]:
df_melt.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8483 entries, 16 to 152517
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   연번                 8483 non-null   int64         
 1   규제시행국              8483 non-null   object        
 2   품목명                8483 non-null   object        
 3   regulation_type    8483 non-null   object        
 4   regulation_status  8483 non-null   object        
 5   target_country     8483 non-null   object        
 6   최종 판정결과(부과기간)      8483 non-null   object        
 7   최종 판정결과(관세율)       7949 non-null   object        
 8   한국대상여부             8483 non-null   object        
 9   hs_code            8483 non-null   object        
 10  hs_4digit          8483 non-null   object        
 11  is_korea_target    8483 non-null   int64         
 12  start_date         8483 non-null   datetime64[ns]
 13  end_date           7331 non-null   datetime64[ns]
 14  date_type 

In [151]:
keep_cols = [
    '연번',
    '규제시행국',
    'hs_4digit',
    'target_country',
    'start_ym',
    'start_date',
    'end_date',
    'date_type',
    'regulation_type',
    'regulation_status',
    'is_korea_target',
    '최종 판정결과(관세율)',
]

In [158]:
df_final = df_melt[keep_cols].rename(columns={
    '연번': 'case_id',
    '규제시행국': 'imposing_country',
    '최종 판정결과(관세율)': 'tariff_clean'
})

In [159]:
print(f"최종 행 수: {len(df_final)}행")
print(f"\nis_korea_target 분포:")
print(df_final['is_korea_target'].value_counts())
print(f"클래스 비율 1:{df_final['is_korea_target'].eq(0).sum() / df_final['is_korea_target'].sum():.1f}")
print(f"\n결측값:")
print(df_final.isnull().sum())

최종 행 수: 8483행

is_korea_target 분포:
is_korea_target
0    4629
1    3854
Name: count, dtype: int64
클래스 비율 1:1.2

결측값:
case_id                 0
imposing_country        0
hs_4digit               0
target_country          0
start_ym                0
start_date              0
end_date             1152
date_type               0
regulation_type         0
regulation_status       0
is_korea_target         0
tariff_clean          534
dtype: int64


In [160]:
df_final.to_csv('kotra_cleaned.csv', index=False, encoding='utf-8-sig')
print("\n✅ kotra_cleaned.csv 저장 완료 — KOTRA 전처리 완료")


✅ kotra_cleaned.csv 저장 완료 — KOTRA 전처리 완료


In [161]:
df_final['imposing_country'].value_counts()

imposing_country
US    3076
CA    1299
GB     825
TH     753
EU     583
TR     336
TW     242
MY     188
AU     172
VN     131
MX     131
PK     124
ID     123
UA     113
CN      86
EG      66
BR      38
IN      30
NZ      30
RU      20
AM      20
KG      20
KZ      20
BY      20
DO      15
CO      13
MA       6
JP       2
ZA       1
Name: count, dtype: int64